# Model Evaluation & Selection

The objective is to compare the performance of Logistic Regression, Decision Tree, Random Forest, and Gradient Boosting models using multiple classification metrics and select the best-performing model for placement prediction.

## Section 1: Import Required Libraries

The required libraries are imported for loading trained models, generating predictions, calculating evaluation metrics, and visualizing model performance.

In [54]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

## Section 2: Load Test Data

The test dataset is loaded for evaluating the trained classification models.

The test data was kept separate from the training data during Phase 6, allowing us to measure how well the models generalize to unseen student records.

In [7]:
df = pd.read_csv("../data/processed/student_placement_features.csv")
df.shape

(100000, 21)

In [8]:
X = df.drop(columns=["placement_status"])
y = df["placement_status"]

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
X_test.shape, y_test.shape

((20000, 20), (20000,))

## Section 3: Transform Test Data

The saved preprocessing pipeline from Phase 5 is loaded and applied to the test data.

The same preprocessing configuration used during model training must be applied to the test data to ensure that the models receive features in the same format.

The test data should be transformed from 20 original features into 28 encoded features.

In [11]:
preprocessor = joblib.load("../models/preprocessor.pkl")

In [12]:
X_test_encoded = preprocessor.transform(X_test)

In [13]:
X_test_encoded.shape

(20000, 28)

## Section 4: Load Trained Models

The trained classification models saved during Phase 6 are loaded for evaluation.

The four models being evaluated are:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting

Loading the saved models allows us to evaluate them on the unseen test dataset without retraining them.

In [14]:
logistic_model = joblib.load("../models/logistic_model.pkl")
decision_tree_model = joblib.load("../models/decision_tree_model.pkl")
random_forest_model = joblib.load("../models/random_forest_model.pkl")
gradient_boosting_model = joblib.load("../models/gradient_boosting_model.pkl")

In [15]:
type(random_forest_model)

sklearn.ensemble._forest.RandomForestClassifier

## Section 5: Generate Predictions

The trained models are used to predict placement status for the unseen test dataset.

Each model generates a binary prediction:

- `0` → Not Placed
- `1` → Placed

The predictions will be used in the following sections to calculate and compare model evaluation metrics.

In [19]:
logistic_predictions = logistic_model.predict(X_test_encoded)
decision_tree_predictions = decision_tree_model.predict(X_test_encoded)
random_forest_predictions = random_forest_model.predict(X_test_encoded)
gradient_boosting_predictions = gradient_boosting_model.predict(X_test_encoded)

In [21]:
logistic_predictions.shape

(20000,)

In [22]:
logistic_predictions[:10]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

In [23]:
decision_tree_predictions.shape

(20000,)

In [24]:
random_forest_predictions.shape

(20000,)

In [25]:
gradient_boosting_predictions.shape

(20000,)

### Observation:

- All four trained classification models generated predictions for the 20,000 unseen test samples.
- Each prediction is a binary value:
  - `0` → Not Placed
  - `1` → Placed
- The predictions will be compared with the actual `y_test` values to calculate model performance metrics.

## Section 6: Calculate Classification Metrics

The predictions generated by the four classification models are compared with the actual placement status of the test dataset.

The following evaluation metrics are calculated:

- **Accuracy:** Measures the proportion of correct predictions.
- **Precision:** Measures how many students predicted as placed were actually placed.
- **Recall:** Measures how many actually placed students were correctly identified.
- **F1-score:** Provides a balance between precision and recall.

These metrics will be used to compare the performance of the four models.

In [40]:
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
logistic_accuracy

0.6995

In [37]:
logistic_precision = precision_score(y_test, logistic_predictions)
logistic_precision

0.7167842031029619

In [38]:
logistic_recall = recall_score(y_test, logistic_predictions)
logistic_recall

0.927710843373494

In [39]:
logistic_f1 = f1_score(y_test, logistic_predictions)
logistic_f1

0.8087205601527689

In [41]:
decision_tree_accuracy = accuracy_score(y_test, decision_tree_predictions)
decision_tree_accuracy

0.59905

In [42]:
decision_tree_precision = precision_score(y_test, decision_tree_predictions)
decision_tree_precision

0.7128074385122976

In [43]:
decision_tree_recall = recall_score(y_test, decision_tree_predictions)
decision_tree_recall

0.6941219423147134

In [45]:
decision_tree_f1 = f1_score(y_test, decision_tree_predictions)
decision_tree_f1

0.7033406089304872

In [46]:
random_forest_accuracy = accuracy_score(y_test, random_forest_predictions)
random_forest_accuracy

0.6926

In [47]:
random_forest_precision = precision_score(y_test, random_forest_predictions)
random_forest_precision

0.7137839215908447

In [48]:
random_forest_recall = recall_score(y_test, random_forest_predictions)
random_forest_recall

0.9199707922599489

In [49]:
random_forest_f1 = f1_score(y_test, random_forest_predictions)
random_forest_f1

0.80386652204428

In [50]:
gradient_boosting_accuracy = accuracy_score(y_test, gradient_boosting_predictions)
gradient_boosting_accuracy

0.69735

In [51]:
gradient_boosting_precision = precision_score(y_test, gradient_boosting_predictions)
gradient_boosting_precision

0.7092781246576843

In [52]:
gradient_boosting_recall = recall_score(y_test, gradient_boosting_predictions)
gradient_boosting_recall

0.945600584154801

In [53]:
gradient_boosting_f1 = f1_score(y_test, gradient_boosting_predictions)
gradient_boosting_f1

0.8105655181047163

## Section 7: Confusion Matrix Analysis

A confusion matrix provides a detailed view of the classification results by comparing the actual placement status with the predicted placement status.

The four outcomes are:

- **True Negative (TN):** Student was not placed and was correctly predicted as Not Placed.
- **False Positive (FP):** Student was not placed but was incorrectly predicted as Placed.
- **False Negative (FN):** Student was placed but was incorrectly predicted as Not Placed.
- **True Positive (TP):** Student was placed and was correctly predicted as Placed.

For this project, false negatives are particularly important because they represent students who were actually placed but were predicted as Not Placed.

In [55]:
logistic_cm = confusion_matrix(y_test, logistic_predictions)
logistic_cm

array([[ 1285,  5020],
       [  990, 12705]])

In [56]:
decision_tree_cm = confusion_matrix(y_test, decision_tree_predictions)
decision_tree_cm

array([[2475, 3830],
       [4189, 9506]])

In [57]:
random_forest_cm = confusion_matrix(y_test, random_forest_predictions)
random_forest_cm

array([[ 1253,  5052],
       [ 1096, 12599]])

In [58]:
gradient_boosting_cm = confusion_matrix(y_test, gradient_boosting_predictions)
gradient_boosting_cm

array([[  997,  5308],
       [  745, 12950]])

## Confusion Matrix Comparison

Gradient Boosting produced the lowest number of false negatives (745) and the highest number of true positives (12,950) among the four evaluated models.

Since the project aims to identify students who are likely to be placed, reducing false negatives is important. Therefore, Gradient Boosting shows the strongest performance for identifying placed students.

However, false positives are also considered when interpreting the model, which is why multiple evaluation metrics are used rather than relying on a single metric.

In [59]:
print(classification_report(
    y_test,
    gradient_boosting_predictions,
    target_names=["Not Placed", "Placed"]
))

              precision    recall  f1-score   support

  Not Placed       0.57      0.16      0.25      6305
      Placed       0.71      0.95      0.81     13695

    accuracy                           0.70     20000
   macro avg       0.64      0.55      0.53     20000
weighted avg       0.67      0.70      0.63     20000



## Section 8: ROC-AUC Evaluation

ROC-AUC is used to measure how well each classification model distinguishes between the two placement classes across different classification thresholds.

A higher ROC-AUC indicates better ability to distinguish between students who are placed and students who are not placed.

ROC-AUC will be considered alongside accuracy, precision, recall, F1-score, and confusion matrix results when selecting the final model.

In [60]:
gradient_boosting_probabilities = gradient_boosting_model.predict_proba(
    X_test_encoded
)[:, 1]

gradient_boosting_roc_auc = roc_auc_score(
    y_test,
    gradient_boosting_probabilities
)

gradient_boosting_roc_auc

0.6819927507593637

In [61]:
logistic_probabilities = logistic_model.predict_proba(
    X_test_encoded
)[:, 1]

logistic_roc_auc = roc_auc_score(
    y_test,
    logistic_probabilities
)

logistic_roc_auc

0.6856072607060062

In [62]:
decision_tree_probabilities = decision_tree_model.predict_proba(
    X_test_encoded
)[:, 1]

decision_tree_roc_auc = roc_auc_score(
    y_test,
    decision_tree_probabilities
)

decision_tree_roc_auc

0.5433337705229396

In [63]:
random_forest_probabilities = random_forest_model.predict_proba(
    X_test_encoded
)[:, 1]

random_forest_roc_auc = roc_auc_score(
    y_test,
    random_forest_probabilities
)

random_forest_roc_auc

0.666513916671661

## Section 9: Model Comparison

The performance of all four classification models is compared using accuracy, precision, recall, F1-score, and ROC-AUC.

The comparison helps identify the most suitable model for the student placement prediction system.

Since the primary objective is to identify students who are likely to be placed, recall, F1-score, and false-negative count are given particular consideration along with the other evaluation metrics.

In [64]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
        logistic_accuracy,
        decision_tree_accuracy,
        random_forest_accuracy,
        gradient_boosting_accuracy
    ],
    "Precision": [
        logistic_precision,
        decision_tree_precision,
        random_forest_precision,
        gradient_boosting_precision
    ],
    "Recall": [
        logistic_recall,
        decision_tree_recall,
        random_forest_recall,
        gradient_boosting_recall
    ],
    "F1 Score": [
        logistic_f1,
        decision_tree_f1,
        random_forest_f1,
        gradient_boosting_f1
    ],
    "ROC-AUC": [
        logistic_roc_auc,
        decision_tree_roc_auc,
        random_forest_roc_auc,
        gradient_boosting_roc_auc
    ]
})

In [65]:
model_comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.69950,0.716784,0.927711,0.808721,0.685607
1,Decision Tree,0.59905,0.712807,0.694122,0.703341,0.543334
2,Random Forest,0.69260,0.713784,0.919971,0.803867,0.666514
3,Gradient Boosting,0.69735,0.709278,0.945601,0.810566,0.681993


## Section 10: Final Model Selection

Based on the evaluation results, Gradient Boosting was selected as the final model for the student placement prediction system.

Gradient Boosting achieved the highest recall (94.56%) and highest F1-score (81.06%) among the evaluated models. It also produced the lowest number of false negatives (745) and the highest number of true positives (12,950).

Although Logistic Regression achieved slightly higher precision and ROC-AUC, Gradient Boosting was preferred because the primary objective of this project is to identify students who are likely to be placed and minimize false-negative predictions.

### Selected Model:
**Gradient Boosting Classifier**

### Important Limitation:
The selected model has relatively low recall for the Not Placed class (16%). This indicates that the model tends to predict students as Placed more frequently and has difficulty identifying students who are actually Not Placed.

## Section 11: Save Selected Model

The selected Gradient Boosting model is saved as the final placement prediction model.

The saved model will be loaded in Phase 8 to generate placement predictions for new student data.

In [66]:
joblib.dump(
    gradient_boosting_model,
    "../models/best_model.pkl"
)

['../models/best_model.pkl']

In [67]:
import os

os.path.exists("../models/best_model.pkl")

True